In [13]:
from pathlib import Path

# Get project root
try:
    PROJECT_ROOT = Path(__file__).parent.parent
except NameError:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

data_processed = PROJECT_ROOT / "data" / "processed"

print("PROJECT ROOT:", PROJECT_ROOT)

PROJECT ROOT: /workspaces/flood-ml-research


In [14]:
import pandas as pd

df = pd.read_parquet(data_processed / "makassar_labeled.parquet")

print("=== DATA LOADED ===")
print("Shape:", df.shape)

df.head()

=== DATA LOADED ===
Shape: (5174884, 6)


,grid_id,date,rainfall,extreme_rain,flood_label,month
0,227,2018-01-01,23.936794,0,0,1
1,227,2018-01-02,14.944647,0,0,1
2,227,2018-01-03,13.822836,0,0,1
3,227,2018-01-04,13.823023,0,0,1
4,227,2018-01-05,46.497144,0,0,1


In [15]:
df["date"] = pd.to_datetime(df["date"])

In [16]:
df = df.sort_values(["grid_id", "date"])

print("=== DATA SORTED ===")

=== DATA SORTED ===


In [17]:
lags = [1, 3, 7, 14, 30]

for lag in lags:
    df[f"rain_lag_{lag}"] = df.groupby("grid_id")["rainfall"].shift(lag)

print("=== LAG FEATURES CREATED ===")

=== LAG FEATURES CREATED ===


In [18]:
df["rain_7d_sum"] = (
    df.groupby("grid_id")["rainfall"]
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

df["rain_30d_sum"] = (
    df.groupby("grid_id")["rainfall"]
    .rolling(30)
    .sum()
    .reset_index(level=0, drop=True)
)

print("=== ROLLING FEATURES CREATED ===")

=== ROLLING FEATURES CREATED ===


In [19]:
before_shape = df.shape

print("=== SHAPE BEFORE DROPNA ===")
print(before_shape)

=== SHAPE BEFORE DROPNA ===
(5174884, 13)


In [20]:
df = df.dropna()

after_shape = df.shape

print("\n=== SHAPE AFTER DROPNA ===")
print(after_shape)

print("\n=== ROWS DROPPED ===")
print(before_shape[0] - after_shape[0])


=== SHAPE AFTER DROPNA ===
(5089864, 13)

=== ROWS DROPPED ===
85020


In [21]:
print("=== SAMPLE DATA ===")
df.head(10)

=== SAMPLE DATA ===


,grid_id,date,rainfall,extreme_rain,flood_label,month,rain_lag_1,rain_lag_3,rain_lag_7,rain_lag_14,rain_lag_30,rain_7d_sum,rain_30d_sum
30,227,2018-01-31,21.210439,0,0,1,34.377993,13.073814,19.339504,10.052845,23.936794,142.391031,537.612650
31,227,2018-02-01,12.173188,0,0,2,21.210439,11.399098,28.133538,13.177414,14.944647,126.430681,534.841191
32,227,2018-02-02,9.366065,0,0,2,12.173188,34.377993,18.978356,16.493025,13.822836,116.818390,530.384420
33,227,2018-02-03,25.876190,0,0,2,9.366065,21.210439,15.217792,6.401373,13.823023,127.476788,542.437587
34,227,2018-02-04,33.834417,0,0,2,25.876190,12.173188,13.073814,29.700601,46.497144,148.237390,529.774860
35,227,2018-02-05,31.812098,0,0,2,33.834417,9.366065,11.399098,5.351814,28.667062,168.650390,532.919895
36,227,2018-02-06,8.010470,0,0,2,31.812098,25.876190,34.377993,4.727934,11.310780,142.282867,529.619585
37,227,2018-02-07,12.984978,0,0,2,8.010470,33.834417,21.210439,19.339504,24.698145,134.057406,517.906419
38,227,2018-02-08,11.214350,0,0,2,12.984978,31.812098,12.173188,28.133538,19.989603,133.098568,509.131166
39,227,2018-02-09,14.382881,0,0,2,11.214350,8.010470,9.366065,18.978356,2.159149,138.115383,521.354898


In [22]:
print("=== RANDOM SAMPLE ===")
df.sample(10, random_state=42)

=== RANDOM SAMPLE ===


,grid_id,date,rainfall,extreme_rain,flood_label,month,rain_lag_1,rain_lag_3,rain_lag_7,rain_lag_14,rain_lag_30,rain_7d_sum,rain_30d_sum
799187,17411,2021-05-10,11.399314,0,0,5,19.610529,16.420853,20.902389,24.109399,6.241061,190.743930,619.438036
2304378,19724,2022-11-28,12.080080,0,0,11,7.120431,17.764304,15.150819,20.666891,6.399512,138.249496,551.829845
4811644,22953,2018-05-15,14.167710,0,0,5,7.537717,15.826356,11.724048,26.978148,16.787472,81.241458,578.315824
1060258,17854,2021-03-24,12.253458,0,0,3,46.474717,32.708440,3.203843,4.282588,17.862885,172.694536,643.889336
4999190,23302,2021-11-29,21.193164,0,0,11,15.036641,4.205100,21.858181,41.765445,7.862432,98.097318,532.484515
3001975,20647,2018-02-01,23.966561,0,0,2,19.104115,19.890957,12.508238,15.562868,8.668361,104.967132,541.027652
4672230,22718,2021-08-15,19.765998,0,0,8,25.282734,29.268447,62.564961,13.604610,9.307795,151.129003,591.149099
1246997,18107,2022-07-24,47.209868,0,0,7,8.407388,32.705534,63.167411,30.575231,26.437066,139.222858,639.216107
1282172,18203,2018-11-17,17.798845,0,0,11,12.894778,35.590382,14.296618,36.215326,22.027527,154.818295,584.348659
33634,15769,2020-02-06,28.416772,0,0,2,31.315540,12.652145,82.930466,30.431630,17.329861,155.126511,760.221105


In [23]:
print("=== FEATURE COLUMNS ===")
print(df.columns)

=== FEATURE COLUMNS ===
Index(['grid_id', 'date', 'rainfall', 'extreme_rain', 'flood_label', 'month',
       'rain_lag_1', 'rain_lag_3', 'rain_lag_7', 'rain_lag_14', 'rain_lag_30',
       'rain_7d_sum', 'rain_30d_sum'],
      dtype='object')


In [24]:
out_path = data_processed / "makassar_features.parquet"

df.to_parquet(out_path, index=False)

print("✔ FEATURES SAVED:", out_path)

✔ FEATURES SAVED: /workspaces/flood-ml-research/data/processed/makassar_features.parquet
